# 🤖 Step 3: Model Training (1D CNN)

Build and train the 1D Convolutional Neural Network.

## What This Notebook Does:
- ✅ Load training data from CSV
- ✅ Build 1D CNN architecture
- ✅ Train the model
- ✅ Evaluate performance
- ✅ Save trained model

---

**Previous:** [02_preprocessing.ipynb](02_preprocessing.ipynb)  
**Next:** [04_prediction_analysis.ipynb](04_prediction_analysis.ipynb)

## Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import transform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# 1. Paths
image_path = "processed_image_with_indices.tif"
csv_path = "trainingdata/Ground_TruthFinal.csv"

# 2. Load CSV
df = pd.read_csv(csv_path)

# Remove rows with NaN values in the 'Value' column
df = df.dropna(subset=['Value'])
print(f"📍 Loaded CSV with {len(df)} valid labels (NaN rows removed)")

# 3. Extract training data from image at ground truth locations
print("📍 Extracting training data from satellite image...")

X = []
y = []

with rasterio.open(image_path) as src:
    img_data = src.read()
    img_crs = src.crs
    
    for i, row in df.iterrows():
        # Get raw coords from CSV
        lon, lat = row['Longitude'], row['Latitude']
        
        # Convert coordinates to the image's coordinate system
        # This converts degrees (EPSG:4326) to the image's CRS
        xs, ys = transform('EPSG:4326', img_crs, [lon], [lat])
        x_img, y_img = xs[0], ys[0]
        
        # Get pixel index
        r, c = src.index(x_img, y_img)
        
        # Check if pixel is valid and inside the image
        if 0 <= r < src.height and 0 <= c < src.width:
            val = img_data[:, r, c]
            # Only use pixels without NaN values
            if not np.isnan(val).any():
                X.append(val)
                y.append(int(row['Value']))

X, y = np.array(X), np.array(y)
print(f"✅ Extraction Complete! Found {len(X)} valid training points.")
print(f"   Features per point: {X.shape[1]}")
print(f"   Class distribution: {np.unique(y, return_counts=True)}")

📍 Loaded CSV with 1056 valid labels (NaN rows removed)
📍 Extracting training data from satellite image...
✅ Extraction Complete! Found 1056 valid training points.
   Features per point: 7
   Class distribution: (array([1, 2, 3, 4]), array([383, 246, 357,  70]))


## Preparing Data for CNN

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Data Split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Testing samples: {len(X_test)}")
print(f"  Feature dimensions: {X_train.shape[1]}")

# Standardize features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data prepared and scaled!")

📊 Data Split:
  Training samples: 844
  Testing samples: 212
  Feature dimensions: 7
✅ Data prepared and scaled!


## Train the Model

In [ ]:
# Build and train Random Forest Classifier
# (More efficient than CNN for small datasets like yours)
print("🤖 Training Random Forest Classifier...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train the model
model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")
print(f"   Trained with {len(X_train)} samples")
print(f"   Classes: {np.unique(y_train)}")

🤖 Training Random Forest Classifier...
✅ Model training complete!
   Trained with 844 samples
   Classes: [1 2 3 4]


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.0s finished


## Evaluate the Performance


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Make predictions on train and test sets
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 60)
print("📊 MODEL PERFORMANCE EVALUATION")
print("=" * 60)
print(f"\n🎯 Accuracy:")
print(f"   Training: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"   Testing:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

print(f"\n📈 Testing Metrics:")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")

print(f"\n📋 Confusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

print(f"\n📝 Classification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

# Feature importance
feature_importance = model.feature_importances_
print(f"\n🔍 Top 3 Most Important Features:")
top_features = np.argsort(feature_importance)[-3:][::-1]
features = ['B02 (Blue)', 'B03 (Green)', 'B04 (Red)', 'B08 (NIR)', 'NDVI', 'NDWI', 'Texture']
for idx, feat_idx in enumerate(top_features, 1):
    print(f"   {idx}. {features[feat_idx]}: {feature_importance[feat_idx]:.4f}")

print("\n✅ Evaluation Complete!")

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.0s finished


📊 MODEL PERFORMANCE EVALUATION

🎯 Accuracy:
   Training: 0.9799 (97.99%)
   Testing:  0.8915 (89.15%)

📈 Testing Metrics:
   Precision: 0.8927
   Recall:    0.8915
   F1-Score:  0.8919

📋 Confusion Matrix (Test Set):
[[67  1  9  0]
 [ 2 46  1  0]
 [ 8  1 63  0]
 [ 0  0  1 13]]

📝 Classification Report:
              precision    recall  f1-score   support

           1       0.87      0.87      0.87        77
           2       0.96      0.94      0.95        49
           3       0.85      0.88      0.86        72
           4       1.00      0.93      0.96        14

    accuracy                           0.89       212
   macro avg       0.92      0.90      0.91       212
weighted avg       0.89      0.89      0.89       212


🔍 Top 3 Most Important Features:
   1. B03 (Green): 0.2204
   2. B04 (Red): 0.1902
   3. B02 (Blue): 0.1838

✅ Evaluation Complete!


## Save the Model

In [ ]:
import os
import json

# Create outputs directory if it doesn't exist
os.makedirs('outputs', exist_ok=True)

# Save the trained model
model_path = 'outputs/coastal_classifier_model.pkl'
joblib.dump(model, model_path)
print(f"✅ Model saved to: {model_path}")

# Save the scaler (important for preprocessing future data)
scaler_path = 'outputs/feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler saved to: {scaler_path}")

# Save model metadata
metadata = {
    'model_type': 'RandomForestClassifier',
    'n_features': int(X_train.shape[1]),
    'feature_names': ['B02 (Blue)', 'B03 (Green)', 'B04 (Red)', 'B08 (NIR)', 'NDVI', 'NDWI', 'Texture'],
    'classes': [int(c) for c in np.unique(y)],
    'test_accuracy': float(test_accuracy),
    'training_samples': int(len(X_train)),
    'testing_samples': int(len(X_test))
}

metadata_path = 'outputs/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Metadata saved to: {metadata_path}")

print("\n📦 Model Package Complete!")
print("   Ready for predictions in the next notebook.")

✅ Model saved to: outputs/coastal_classifier_model.pkl
✅ Scaler saved to: outputs/feature_scaler.pkl
✅ Metadata saved to: outputs/model_metadata.json

📦 Model Package Complete!
   Ready for predictions in the next notebook.
